# 06 — Bytecode CPython et le module `dis`

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :
- expliquer ce qu'est le bytecode CPython et où il se situe dans la chaîne de compilation ;
- utiliser `dis.dis()`, `dis.Bytecode`, et `dis.code_info()` pour inspecter du code ;
- lire les instructions bytecode courantes (`LOAD_FAST`, `CALL`, `BINARY_OP`, etc.) ;
- comprendre les optimisations du compilateur (constant folding, PEP 709 inline comprehensions) ;
- analyser l'objet `code` (`__code__`) et ses attributs (`co_consts`, `co_varnames`, etc.).

## Prérequis — ce que vous connaissez déjà

Ce notebook s'adresse à un développeur Python **confirmé**. Vous maîtrisez déjà :
- le modèle objet complet (classes, méthodes, héritage, MRO) ;
- les décorateurs, les descripteurs, les métaclasses (notebooks 01 à 05 de la section Métaprogrammation) ;
- les closures, les générateurs, les compréhensions ;
- les fonctions intégrées (`compile`, `exec`, `eval`) ;
- `type()` dynamique et manipulation des espaces de noms.

## Plan

1. Où se situe le bytecode dans CPython ?
2. Premier désassemblage avec `dis.dis()`
3. Anatomie d'une instruction bytecode
4. L'objet `code` et ses attributs
5. `dis.Bytecode` — itérer sur les instructions
6. `dis.code_info()` — résumé complet
7. Instructions courantes — lexique
8. Optimisations du compilateur
9. PEP 709 — compréhensions inline (Python 3.12+)
10. Comparaison de versions de code
11. Synthèse
12. Exercices
13. Ressources

---

## 1. Où se situe le bytecode dans CPython ?

Quand vous exécutez un script Python, le code source passe par **trois étapes** :

| Étape | Entrée | Sortie |
|---|---|---|
| **Parsing** | texte `.py` | AST (arbre syntaxique) |
| **Compilation** | AST | bytecode (objet `code`) |
| **Exécution** | bytecode | résultat via la VM CPython |

Le bytecode est un jeu d'instructions **propre à CPython**. Il n'est **pas portable** entre les versions de Python : les opcodes changent régulièrement.

In [ ]:
import sys
print(f"Version CPython : {sys.version}")

Le fichier `.pyc` que vous trouvez dans `__pycache__/` est la sérialisation (marshalling) du bytecode compilé. Il évite de recompiler le source à chaque exécution.

---

## 2. Premier désassemblage avec `dis.dis()`

Le module `dis` (pour *disassemble*) fait partie de la bibliothèque standard. Sa fonction principale, `dis.dis()`, affiche le bytecode d'un objet compilé.

In [ ]:
import dis

In [ ]:
def addition(a, b):
    return a + b

In [ ]:
dis.dis(addition)

Chaque ligne affiche :
- le **numéro de ligne** du source ;
- l'**offset** (position en octets dans le bytecode) ;
- le **nom de l'opcode** ;
- l'**argument** éventuel (index dans `co_varnames`, `co_consts`, etc.) ;
- la **valeur interprétée** de l'argument entre parenthèses.

### Désassembler une expression simple

In [ ]:
dis.dis("x = 2 + 3")

Vous voyez ici le **constant folding** : le compilateur a remplacé `2 + 3` par `5` directement. L'instruction `LOAD_CONST` charge 5, pas 2 et 3 séparément.

### Désassembler une lambda

In [ ]:
dis.dis(lambda x: x ** 2)

---

## 3. Anatomie d'une instruction bytecode

Depuis Python 3.6, chaque instruction occupe exactement **2 octets** (word code) :
- 1 octet pour l'**opcode** (identifiant de l'instruction) ;
- 1 octet pour l'**argument** (ou 0 si pas d'argument).

Pour les arguments > 255, un opcode spécial `EXTENDED_ARG` préfixe l'instruction.

In [ ]:
import opcode

# Nombre total d'opcodes définis
print(f"Nombre d'opcodes : {len([o for o in opcode.opname if o.startswith('<') is False])}")

In [ ]:
# Quelques opcodes et leur numéro
for name in ["LOAD_FAST", "STORE_FAST", "BINARY_OP", "RETURN_VALUE", "CALL"]:
    if name in opcode.opmap:
        print(f"{name:20s} → opcode {opcode.opmap[name]}")

### Opcodes avec et sans argument

Les opcodes dont la valeur numérique est inférieure à `opcode.HAVE_ARGUMENT` n'ont pas d'argument.

In [ ]:
print(f"HAVE_ARGUMENT = {opcode.HAVE_ARGUMENT}")
print(f"RETURN_VALUE ({opcode.opmap.get('RETURN_VALUE', '?')}) < {opcode.HAVE_ARGUMENT} → pas d'argument")

---

## 4. L'objet `code` et ses attributs

Chaque fonction possède un objet `code` accessible via `__code__`. Cet objet contient tout ce dont la VM a besoin pour exécuter la fonction.

In [ ]:
def exemple(x, y=10):
    z = x + y
    return z * 2

In [ ]:
co = exemple.__code__
print(f"Nom            : {co.co_name}")
print(f"Fichier        : {co.co_filename}")
print(f"Nb arguments   : {co.co_argcount}")
print(f"Nb variables   : {co.co_nlocals}")
print(f"Variables      : {co.co_varnames}")
print(f"Constantes     : {co.co_consts}")
print(f"Stack max      : {co.co_stacksize}")

| Attribut | Description |
|---|---|
| `co_name` | Nom de la fonction |
| `co_varnames` | Variables locales (arguments inclus) |
| `co_consts` | Constantes littérales utilisées |
| `co_names` | Noms globaux référencés |
| `co_freevars` | Variables libres (closures) |
| `co_cellvars` | Variables capturées par des fonctions internes |
| `co_code` | Bytecode brut (bytes) |
| `co_stacksize` | Taille maximale de la pile |

### Bytecode brut

In [ ]:
print(f"Bytecode brut : {co.co_code.hex()}")
print(f"Taille : {len(co.co_code)} octets")

### Closures et `co_freevars`

In [ ]:
def externe():
    facteur = 3
    def interne(x):
        return x * facteur
    return interne

fn = externe()
print(f"co_freevars : {fn.__code__.co_freevars}")

In [ ]:
dis.dis(fn)

---

## 5. `dis.Bytecode` — itérer sur les instructions

`dis.Bytecode` crée un itérable d'objets `Instruction`, plus pratique que la sortie texte de `dis.dis()`.

In [ ]:
def carre(n):
    return n * n

In [ ]:
bc = dis.Bytecode(carre)
for instr in bc:
    print(f"offset={instr.offset:3d}  opname={instr.opname:20s}  arg={instr.arg}  argval={instr.argval}")

### Compter les types d'instructions

In [ ]:
from collections import Counter

def fibonacci(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

compteur = Counter(instr.opname for instr in dis.Bytecode(fibonacci))
for op, count in compteur.most_common(10):
    print(f"{op:25s} {count}")

### Filtrer les appels de fonction

In [ ]:
def complexe(lst):
    return sorted(set(lst), key=len)

appels = [instr for instr in dis.Bytecode(complexe) if "CALL" in instr.opname]
for a in appels:
    print(a)

---

## 6. `dis.code_info()` — résumé complet

`dis.code_info()` affiche un résumé textuel de l'objet `code` : arguments, constantes, variables, etc.

In [ ]:
print(dis.code_info(fibonacci))

---

## 7. Instructions courantes — lexique

| Instruction | Catégorie | Description |
|---|---|---|
| `LOAD_FAST` | Variables | Charge une variable locale sur la pile |
| `STORE_FAST` | Variables | Stocke le sommet de la pile dans une variable locale |
| `LOAD_CONST` | Constantes | Charge une constante (`co_consts[i]`) |
| `LOAD_GLOBAL` | Globales | Charge une variable globale ou builtin |
| `LOAD_ATTR` | Attributs | Charge un attribut d'objet |
| `BINARY_OP` | Opérations | Opération binaire (+, -, *, etc.) |
| `COMPARE_OP` | Comparaisons | Comparaison (==, <, >=, etc.) |
| `CALL` | Appels | Appelle une callable |
| `RETURN_VALUE` | Contrôle | Retourne la valeur au sommet de la pile |
| `POP_JUMP_IF_FALSE` | Sauts | Saut conditionnel |
| `JUMP_FORWARD` | Sauts | Saut inconditionnel vers l'avant |
| `GET_ITER` / `FOR_ITER` | Itération | Protocole itérateur |
| `BUILD_LIST` / `BUILD_TUPLE` | Construction | Crée un conteneur |
| `UNPACK_SEQUENCE` | Déballage | Déballe un itérable |

### Exemple : boucle for

In [ ]:
def somme_carres(n):
    total = 0
    for i in range(n):
        total += i * i
    return total

dis.dis(somme_carres)

---

## 8. Optimisations du compilateur

Le compilateur CPython applique plusieurs optimisations **au moment de la compilation**, visibles dans le bytecode.

### 8.1. Constant folding

In [ ]:
def avec_constantes():
    return 3600 * 24 * 365

# Le compilateur précalcule la constante
print(avec_constantes.__code__.co_consts)

In [ ]:
dis.dis(avec_constantes)

### 8.2. Peephole : chaîne * entier

In [ ]:
def tirets():
    return "-" * 40

print(tirets.__code__.co_consts)

Le compilateur a directement stocké la chaîne de 40 tirets dans les constantes.

### 8.3. Optimisation des branchements

In [ ]:
def test_and(a, b):
    return a and b

dis.dis(test_and)

L'opérateur `and` utilise un saut conditionnel : si `a` est falsy, Python n'évalue même pas `b`. C'est le *short-circuit evaluation* visible dans le bytecode.

### 8.4. `LOAD_FAST` vs `LOAD_GLOBAL`

In [ ]:
import timeit

x_global = 42

def acces_global():
    return x_global

def acces_local():
    x = 42
    return x

print("Global :", timeit.timeit(acces_global, number=1_000_000))
print("Local  :", timeit.timeit(acces_local, number=1_000_000))

L'accès local (`LOAD_FAST`) est plus rapide que l'accès global (`LOAD_GLOBAL`) car il utilise un index direct dans un tableau C, sans recherche dans un dictionnaire.

---

## 9. PEP 709 — compréhensions inline (Python 3.12+)

Avant Python 3.12, chaque compréhension (`[expr for ...]`) créait un **objet `code` séparé** et un appel de fonction implicite. La PEP 709 supprime ce surcoût : la compréhension est désormais compilée **directement dans le bytecode** de la fonction englobante.

In [ ]:
def avant_pep709(lst):
    return [x ** 2 for x in lst]

# Depuis Python 3.12 : plus de code object séparé
print(f"co_consts : {avant_pep709.__code__.co_consts}")
dis.dis(avant_pep709)

**Conséquence pratique** : les compréhensions sont plus rapides (pas de frame supplémentaire), et les variables locales de la compréhension ne « fuient » plus dans la portée englobante (comportement identique, mais mécanisme différent).

### Comparer avec une boucle explicite

In [ ]:
def avec_boucle(lst):
    result = []
    for x in lst:
        result.append(x ** 2)
    return result

print("=== Compréhension ===")
print(f"Instructions : {len(list(dis.Bytecode(avant_pep709)))}")
print()
print("=== Boucle ===")
print(f"Instructions : {len(list(dis.Bytecode(avec_boucle)))}")

---

## 10. Comparaison de versions de code

Un usage pratique de `dis` est de **comparer deux implémentations** d'une même fonction pour choisir la plus efficace.

In [ ]:
def membership_list(val, data):
    return val in data

def membership_set(val, data):
    return val in data

print("=== in list ===")
dis.dis(membership_list)
print()
print("=== in set ===")
dis.dis(membership_set)

Le bytecode est identique ! La différence de performance vient du **type runtime** de `data`, pas du bytecode. Le `in` appelle `__contains__` qui est O(1) pour `set` et O(n) pour `list`. `dis` ne montre que la compilation, pas l'exécution.

### Comparer f-string vs concaténation

In [ ]:
def avec_fstring(nom, age):
    return f"{nom} a {age} ans"

def avec_concat(nom, age):
    return nom + " a " + str(age) + " ans"

print("=== f-string ===")
dis.dis(avec_fstring)
print()
print("=== concaténation ===")
dis.dis(avec_concat)

La f-string utilise `FORMAT_VALUE` et `BUILD_STRING` (opérations optimisées en C), alors que la concaténation crée des objets intermédiaires. C'est l'une des raisons pour lesquelles les f-strings sont plus rapides.

---

## 10bis. `compile()` et manipulation directe

La fonction builtin `compile()` transforme du code source en objet `code` sans l'exécuter.

In [ ]:
source = """
x = 10
y = 20
resultat = x + y
"""

co = compile(source, "<demo>", "exec")
print(f"Type : {type(co)}")
print(f"Constantes : {co.co_consts}")
print(f"Noms : {co.co_names}")

In [ ]:
dis.dis(co)

### `code.replace()` — modifier un objet code

In [ ]:
def multiplie(x, y):
    return x * y

# Changer le nom affiché de la fonction dans le code object
nouveau_co = multiplie.__code__.replace(co_name="produit")
multiplie.__code__ = nouveau_co
print(f"Nom affiché : {multiplie.__code__.co_name}")

---

## 11. Synthèse

| Outil | Usage |
|---|---|
| `dis.dis(obj)` | Affichage texte du bytecode |
| `dis.Bytecode(obj)` | Itérable d'objets `Instruction` |
| `dis.code_info(obj)` | Résumé textuel complet |
| `fn.__code__` | Accès à l'objet `code` |
| `co_consts` / `co_varnames` / `co_names` | Inspection des données compilées |
| `compile(src, file, mode)` | Compilation sans exécution |

**Règles à retenir :**
- Le bytecode est un détail d'implémentation de CPython ; il change entre versions.
- `dis` est un outil de **diagnostic**, pas de production.
- Le compilateur applique des optimisations (constant folding, PEP 709) ; `dis` permet de les observer.
- L'accès local (`LOAD_FAST`) est toujours plus rapide que l'accès global (`LOAD_GLOBAL`).
- Comparer le bytecode ne suffit pas pour comparer les performances : le type runtime compte aussi.

---

## 12. Exercices

### Exercice 1 — Compter les opcodes *(facile)*

Écrire une fonction `compter_opcodes(fn)` qui prend une fonction en argument et retourne un `Counter` des noms d'opcodes présents dans son bytecode.

Testez-la sur la fonction `fibonacci` définie plus haut.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="06_Bytecode_et_dis", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
from collections import Counter
import dis

def compter_opcodes(fn):
    return Counter(instr.opname for instr in dis.Bytecode(fn))

print(compter_opcodes(fibonacci))
```

</details>

### Exercice 2 — Détecter le constant folding *(moyen)*

Écrire une fonction `est_plie(expression: str) -> bool` qui prend une expression Python sous forme de chaîne (ex. `"2 + 3"`) et retourne `True` si le compilateur a réduit l'expression à une seule constante.

*Indice : compilez l'expression avec `compile()` mode `"eval"`, et comptez les instructions.*

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="06_Bytecode_et_dis", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import dis

def est_plie(expression: str) -> bool:
    co = compile(expression, "<test>", "eval")
    instructions = list(dis.Bytecode(co))
    # Si plié : LOAD_CONST + RETURN_VALUE (ou RESUME + LOAD_CONST + RETURN_VALUE)
    opcodes = [i.opname for i in instructions if i.opname != "RESUME"]
    return opcodes == ["LOAD_CONST", "RETURN_VALUE"]

print(est_plie("2 + 3"))         # True
print(est_plie("2 + x"))         # False
print(est_plie("3600 * 24"))     # True
```

</details>

### Exercice 3 — Comparateur de bytecode *(moyen)*

Écrire une fonction `diff_bytecode(fn1, fn2)` qui affiche, côte à côte, les opcodes de deux fonctions. Marquez d'une `*` les lignes qui diffèrent.

Testez avec :
```python
def v1(lst): return [x*2 for x in lst]
def v2(lst): return list(map(lambda x: x*2, lst))
```

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="06_Bytecode_et_dis", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
import dis
from itertools import zip_longest

def diff_bytecode(fn1, fn2):
    bc1 = list(dis.Bytecode(fn1))
    bc2 = list(dis.Bytecode(fn2))
    print(f"{'fn1':30s} | {'fn2':30s}")
    print("-" * 63)
    for i1, i2 in zip_longest(bc1, bc2):
        s1 = f"{i1.opname} {i1.argval}" if i1 else ""
        s2 = f"{i2.opname} {i2.argval}" if i2 else ""
        marker = " *" if s1 != s2 else ""
        print(f"{s1:30s} | {s2:30s}{marker}")

def v1(lst): return [x*2 for x in lst]
def v2(lst): return list(map(lambda x: x*2, lst))

diff_bytecode(v1, v2)
```

</details>

### Exercice 4 — Analyse de performance par le bytecode *(difficile)*

Écrire une fonction `score_complexite(fn) -> dict` qui analyse le bytecode d'une fonction et retourne un dictionnaire avec :
- `"nb_instructions"` : nombre total d'instructions ;
- `"nb_appels"` : nombre d'instructions `CALL*` ;
- `"nb_sauts"` : nombre de sauts conditionnels et inconditionnels ;
- `"nb_load_global"` : nombre de `LOAD_GLOBAL` (potentiellement lents).

Testez sur plusieurs fonctions et comparez.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="06_Bytecode_et_dis", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
import dis

def score_complexite(fn) -> dict:
    instructions = list(dis.Bytecode(fn))
    return {
        "nb_instructions": len(instructions),
        "nb_appels": sum(1 for i in instructions if "CALL" in i.opname),
        "nb_sauts": sum(1 for i in instructions if "JUMP" in i.opname),
        "nb_load_global": sum(1 for i in instructions if i.opname == "LOAD_GLOBAL"),
    }

def tri_builtin(lst):
    return sorted(lst, key=lambda x: -x)

def tri_manuel(lst):
    result = []
    for x in lst:
        inserted = False
        for i, y in enumerate(result):
            if x > y:
                result.insert(i, x)
                inserted = True
                break
        if not inserted:
            result.append(x)
    return result

for fn in [tri_builtin, tri_manuel]:
    print(f"{fn.__name__}: {score_complexite(fn)}")
```

</details>

### Exercice 5 — Réécrire `co_consts` *(deep dive, difficile)*

En utilisant `code.replace()`, écrire une fonction `patch_constante(fn, ancienne, nouvelle)` qui retourne une copie de `fn` où toutes les occurrences de `ancienne` dans `co_consts` ont été remplacées par `nouvelle`.

```python
def salut():
    return "Bonjour le monde"

salut2 = patch_constante(salut, "Bonjour le monde", "Hello world")
print(salut2())  # Hello world
```

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="06_Bytecode_et_dis", exercice=5)


<details>
<summary>📖 Voir la correction</summary>

```python
import types

def patch_constante(fn, ancienne, nouvelle):
    co = fn.__code__
    new_consts = tuple(
        nouvelle if c == ancienne else c
        for c in co.co_consts
    )
    new_code = co.replace(co_consts=new_consts)
    new_fn = types.FunctionType(new_code, fn.__globals__, fn.__name__)
    return new_fn

def salut():
    return "Bonjour le monde"

salut2 = patch_constante(salut, "Bonjour le monde", "Hello world")
print(salut2())  # Hello world
```

</details>

---

## 13. Ressources

- [Module `dis` — documentation officielle](https://docs.python.org/3/library/dis.html)
- [PEP 709 — Inlined Comprehensions](https://peps.python.org/pep-0709/)
- [Module `opcode`](https://docs.python.org/3/library/opcode.html)
- [Inside The Python Virtual Machine](https://leanpub.com/insidethepythonvirtualmachine) — Obi Ike-Nwosu
- [CPython Internals](https://realpython.com/products/cpython-internals-book/) — Anthony Shaw